# 10_GAN_DPS_generation
Ricostruzione limited-angle CT sul test set usando il generatore GAN gia' allenato come prior, guidato dal sinogramma tramite un ciclo di campionamento stile DPS (Diffusion Posterior Sampling), adattato a un GAN invece che a un modello di diffusione. Su Colab con GPU.

## In cosa consiste (ricapitolando)
Il generatore NON prende in input il sinogramma -- prende un vettore latente `z` casuale. Il ciclo qui sotto aggiorna iterativamente `z` per ciascuna slice di test, combinando due spinte:
- **fedelta' ai dati**: gradiente di `||K(G(z)) - y_delta||^2` (K = il vostro CTProjector, y_delta = il sinogramma vero di quella slice)
- **prior**: un termine che tiene `z` in una zona plausibile (dato che il generatore e' stato allenato con `z ~ N(0,1)`, il prior e' semplicemente `||z||^2`, coerente con la log-probabilita' di una gaussiana standard)

piu' un piccolo rumore ad ogni passo (dinamica di Langevin) per fare vero campionamento a posteriori invece di una semplice ottimizzazione deterministica (si puo' disattivare mettendo `NOISE_SCALE = 0`).

Alla fine del ciclo, `G(z)` e' la ricostruzione: plausibile secondo il generatore, coerente con il sinogramma osservato.

## Cosa NON fa questo notebook
Non calcola metriche (PSNR/SSIM), non serve la ground truth qui -- solo generazione, come gia' fatto per Weighted TV. La valutazione va in un notebook separato, sullo stesso schema degli altri metodi.

## Iperparametri -- da considerare punti di partenza, non valori definitivi
`N_ITERS`, `STEP_SIZE`, `DATA_WEIGHT`, `NOISE_SCALE` sono scelti euristicamente qui sotto. Se i risultati non sono soddisfacenti, lo stesso tipo di tuning gia' fatto per Weighted TV (su validation, griglia, PSNR/SSIM) si puo' ripetere qui.

## Prerequisiti su Drive
`sinograms.zip` (gia' presente) + `gan_checkpoints/generator_weights_best.pth` (scritto dal notebook di training).

In [ ]:
# DA ESEGURIE SU COLAB PER GENERARE LE RICOSTRUZIONI GAN_DPS

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

SINOGRAMS_ZIP = "/content/drive/MyDrive/sinograms.zip"
LOCAL_SINOGRAMS_DIR = Path("/content/sinograms")

if not LOCAL_SINOGRAMS_DIR.exists():
    print("Estraggo i sinogrammi...")
    get_ipython().system('unzip -q "{SINOGRAMS_ZIP}" -d /content/')
else:
    print("Sinogrammi gia' estratti, salto.")

GENERATOR_WEIGHTS_PATH = Path("/content/drive/MyDrive/gan_checkpoints/generator_weights_best.pth")
assert GENERATOR_WEIGHTS_PATH.exists(), f"Non trovo {GENERATOR_WEIGHTS_PATH} -- il training del GAN e' finito?"

Mounted at /content/drive
Estraggo i sinogrammi...


In [2]:
# ============================================================
# Installazione IPPy + patch del bug CuPy (stessa patch delle altre volte
# -- serve di nuovo qui perche' torniamo a usare CTProjector)
# ============================================================
get_ipython().system('pip install -q git+https://github.com/devangelista2/IPPy.git')
try:
    get_ipython().system('pip install -q cupy-cuda12x')
    import cupy  # noqa: F401
    print("CuPy installato correttamente.")
except Exception as e:
    print(f"CuPy non disponibile ({e}).")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 15.5 MB/s eta 0:00:00
CuPy installato correttamente.


In [3]:
import re, glob

candidates = glob.glob("/usr/**/dist-packages/IPPy/operators.py", recursive=True) + \
             glob.glob("/usr/**/site-packages/IPPy/operators.py", recursive=True)
assert candidates, "Non trovo operators.py di IPPy installato."
operators_path = candidates[0]

with open(operators_path) as f:
    src = f.read()

pattern = re.compile(
    r'(elif torch\.cuda\.is_available\(\) and not force_cpu:\s*\n'
    r'\s*warnings\.warn\(\s*\n'
    r'\s*"CUDA available but CuPy not found\. CTProjector limited to CPU operations for ASTRA data transfer\."\s*\n'
    r'\s*\)\s*\n'
    r'\s*# Force CPU mode if CuPy isn\'t there for GPU data handling\s*\n'
    r'(\s*)self\.use_gpu = False)'
)

def _fix(m):
    indent = m.group(2)
    return (
        "elif torch.cuda.is_available() and not force_cpu:\n"
        f"{indent}try:\n"
        f"{indent}    import cupy  # noqa: F401\n"
        f"{indent}except ImportError:\n"
        f"{indent}    warnings.warn(\n"
        f'{indent}        "CUDA available but CuPy not found. CTProjector limited to CPU operations for ASTRA data transfer."\n'
        f"{indent}    )\n"
        f"{indent}    self.use_gpu = False\n"
    )

new_src, n = pattern.subn(_fix, src)
if n == 1:
    with open(operators_path, "w") as f:
        f.write(new_src)
    print(f"Patch applicata a {operators_path}")
elif "import cupy  # noqa: F401" in src:
    print("Patch gia' presente, salto.")
else:
    raise RuntimeError(f"Blocco da patchare non trovato in {operators_path}.")

Patch applicata a /usr/local/lib/python3.13/dist-packages/IPPy/operators.py


In [4]:
import math
import time
import shutil

import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm

from IPPy import operators

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo in uso: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# NOTA: CTProjector non ha il path GPU implementato per i tensori PyTorch
# (vedi 05_WeightedTV_generation.ipynb) -- i tensori che passano per K
# restano su CPU per tutto il notebook. Solo il generatore gira su DEVICE.

Dispositivo in uso: cuda
GPU: NVIDIA A100-SXM4-40GB


In [5]:
# ============================================================
# Architettura del generatore (identica a 08/09 -- serve solo il
# generatore qui, il discriminatore non c'entra piu' nulla adesso)
# ============================================================

class Generator(nn.Module):
    def __init__(self, latent_dim=100, ngf=32):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, ngf * 16, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 16), nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 16, ngf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 8), nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),

            nn.ConvTranspose2d(ngf, ngf // 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf // 2), nn.ReLU(True),

            nn.ConvTranspose2d(ngf // 2, 1, 4, 2, 1, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, z):
        return self.net(z.view(z.size(0), self.latent_dim, 1, 1))


LATENT_DIM = 100
NGF = 32

generator = Generator(LATENT_DIM, NGF).to(DEVICE)
generator.load_state_dict(torch.load(GENERATOR_WEIGHTS_PATH, map_location=DEVICE))
generator.eval()
for p in generator.parameters():
    p.requires_grad_(False)   # non alleniamo piu' il generatore, solo z

print(f"Generatore caricato da: {GENERATOR_WEIGHTS_PATH}")

Generatore caricato da: /content/drive/MyDrive/gan_checkpoints/generator_weights_best.pth


In [6]:
# ============================================================
# Geometria e proiettori (stessa di sempre)
# ============================================================
IMG_SIZE = (256, 256)
NOISE_LEVEL = 0.005

ANGLE_CONFIGS = {
    90: np.linspace(-45, 45, 90),
    45: np.linspace(-45, 45, 45),
    30: np.linspace(-30, 30, 32)[1:-1],
    15: np.linspace(-30, 30, 17)[1:-1],
}

PROJECTORS = {
    n_angles: operators.CTProjector(
        img_shape=IMG_SIZE,
        angles=np.deg2rad(angles),
        geometry="parallel",
        force_cpu=False,   # ASTRA usa comunque il proiettore CUDA internamente
    )
    for n_angles, angles in ANGLE_CONFIGS.items()
}

Attempting to create ASTRA projector type: 'cuda' for 'parallel' geometry...
Successfully created ASTRA projector type: 'cuda'
CTProjector initialized. Geometry: parallel. Using GPU: True. FBP Algorithm: FBP_CUDA
Attempting to create ASTRA projector type: 'cuda' for 'parallel' geometry...
Successfully created ASTRA projector type: 'cuda'
CTProjector initialized. Geometry: parallel. Using GPU: True. FBP Algorithm: FBP_CUDA
Attempting to create ASTRA projector type: 'cuda' for 'parallel' geometry...
Successfully created ASTRA projector type: 'cuda'
CTProjector initialized. Geometry: parallel. Using GPU: True. FBP Algorithm: FBP_CUDA
Attempting to create ASTRA projector type: 'cuda' for 'parallel' geometry...
Successfully created ASTRA projector type: 'cuda'
CTProjector initialized. Geometry: parallel. Using GPU: True. FBP Algorithm: FBP_CUDA


In [7]:
# ============================================================
# Ciclo DPS-adattato: aggiorna z per una singola slice di test,
# combinando fedelta' ai dati (attraverso K) e prior su z, con un
# piccolo rumore ad ogni passo (dinamica di Langevin).
# ============================================================

def dps_reconstruct(
    y_delta,          # sinogramma osservato, tensore CPU (1,1,mx,my)
    K,                 # CTProjector di questa configurazione angolare
    n_iters=300,
    data_weight=1.0,
    step_size=1e-2,
    noise_scale=0.01,
    z_init=None,
):
    # Adam come ottimizzatore per z (non discesa a passo fisso -- quella
    # dava risultati instabili nel tuning, vincitore sempre al bordo
    # della griglia). Stessa versione usata in 10a_GAN_DPS_hyperparameter_tuning.ipynb.
    z = torch.randn(1, LATENT_DIM, device=DEVICE, requires_grad=True) if z_init is None else z_init.clone().requires_grad_(True)
    optimizer = torch.optim.Adam([z], lr=step_size)   # step_size = learning rate di Adam

    for _ in range(n_iters):
        optimizer.zero_grad()

        x = generator(z)              # (1,1,256,256), su DEVICE
        x_cpu = x.cpu()                # ponte verso CPU per K -- operazione differenziabile

        residual = K(x_cpu) - y_delta
        fidelity_loss = torch.sum(residual ** 2)
        prior_loss = torch.sum(z ** 2)
        loss = data_weight * fidelity_loss + prior_loss.cpu()

        loss.backward()
        optimizer.step()

        if noise_scale > 0:
            with torch.no_grad():
                z += math.sqrt(2 * step_size) * noise_scale * torch.randn_like(z)

    with torch.no_grad():
        x_final = generator(z)
    return x_final.detach().cpu().numpy(), z.detach()

In [8]:
# ============================================================
# Test rapido su UNA immagine: verifica che tutto funzioni e da' una
# stima del tempo totale prima di lanciare il loop completo.
# ============================================================

import json as _json

N_ITERS = 300

TUNING_RESULT_PATH = Path("/content/drive/MyDrive/gan_dps_tuning/best_params_dps.json")
DEFAULT_PARAMS = {"data_weight": 1.0, "step_size": 1e-2, "noise_scale": 0.01}

if TUNING_RESULT_PATH.exists():
    with open(TUNING_RESULT_PATH) as f:
        best_params_per_config = _json.load(f)
    best_params_per_config = {int(k): v for k, v in best_params_per_config.items()}
    print("Parametri DPS presi dal tuning:")
    for n_angles, p in best_params_per_config.items():
        print(f"  {n_angles} angoli -> data_weight={p['data_weight']}, step_size={p['step_size']}, noise_scale={p['noise_scale']}")
else:
    best_params_per_config = {n: dict(DEFAULT_PARAMS) for n in [90, 45, 30, 15]}
    print(f"File di tuning non trovato, uso i default per tutte le configurazioni: {DEFAULT_PARAMS}")

test_n_angles = 90
DATA_WEIGHT = best_params_per_config[test_n_angles]["data_weight"]
STEP_SIZE = best_params_per_config[test_n_angles]["step_size"]
NOISE_SCALE = best_params_per_config[test_n_angles]["noise_scale"]
sino_paths = sorted((LOCAL_SINOGRAMS_DIR / "test" / str(test_n_angles)).rglob("*.npy"))
sinogram = np.load(sino_paths[0])
y_delta = torch.from_numpy(sinogram).float().unsqueeze(0).unsqueeze(0)   # CPU

torch.manual_seed(123)
t0 = time.time()
x_sol, _ = dps_reconstruct(
    y_delta, PROJECTORS[test_n_angles],
    n_iters=N_ITERS, data_weight=DATA_WEIGHT, step_size=STEP_SIZE, noise_scale=NOISE_SCALE,
)
elapsed = time.time() - t0

n_totale = sum(len(list((LOCAL_SINOGRAMS_DIR / "test" / str(n)).rglob("*.npy"))) for n in ANGLE_CONFIGS)
print(f"Tempo per una immagine ({N_ITERS} iterazioni DPS): {elapsed:.2f} s")
print(f"Range ricostruzione: [{x_sol.min():.4f}, {x_sol.max():.4f}]")
print(f"Stima per l'intero test set ({n_totale} immagini totali, 4 config): {elapsed * n_totale / 60:.1f} minuti")

Parametri DPS presi dal tuning:
  90 angoli -> data_weight=0.01, step_size=0.01, noise_scale=0.0
  45 angoli -> data_weight=0.01, step_size=0.01, noise_scale=0.0
  30 angoli -> data_weight=0.1, step_size=0.001, noise_scale=0.02
  15 angoli -> data_weight=0.1, step_size=0.001, noise_scale=0.02
Tempo per una immagine (300 iterazioni DPS): 2.64 s
Range ricostruzione: [0.0000, 0.8356]
Stima per l'intero test set (1308 immagini totali, 4 config): 57.6 minuti


In [9]:
# ============================================================
# Utility di sincronizzazione incrementale su Drive (stessa di
# 05_WeightedTV_generation.ipynb)
# ============================================================
def sync_to_drive(src_dir: Path, dst_dir: Path):
    if not src_dir.exists():
        return
    dst_dir.mkdir(parents=True, exist_ok=True)
    n_copied = 0
    for f in src_dir.rglob("*.npy"):
        rel = f.relative_to(src_dir)
        dst = dst_dir / rel
        if dst.exists():
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst)
        n_copied += 1
    if n_copied:
        print(f"Sincronizzati {n_copied} nuovi file su Drive ({dst_dir}).")

In [10]:
# ============================================================
# GENERAZIONE sul test set, per le 4 configurazioni. Checkpoint su
# Drive ogni SYNC_EVERY immagini; se un file e' gia' su Drive viene
# saltato (ripresa automatica dopo una disconnessione).
# ============================================================

SYNC_EVERY = 100
SPLIT = "test"

LOCAL_OUTPUT_DIR = Path("/content/gan_dps_reconstructions")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/gan_dps_reconstructions")

failures = []

for n_angles, K in PROJECTORS.items():
    params = best_params_per_config[n_angles]
    data_weight, step_size, noise_scale = params["data_weight"], params["step_size"], params["noise_scale"]
    print(f"\n--> Angoli: {n_angles} | data_weight: {data_weight} | step_size: {step_size} | noise_scale: {noise_scale}")

    input_sino_dir = LOCAL_SINOGRAMS_DIR / SPLIT / str(n_angles)
    local_out_dir = LOCAL_OUTPUT_DIR / SPLIT / str(n_angles)
    drive_out_dir = DRIVE_OUTPUT_DIR / SPLIT / str(n_angles)

    sino_paths = sorted(
        input_sino_dir.rglob("*.npy"),
        key=lambda p: (p.parent.name, int(p.stem))
    )

    n_since_sync = 0

    for sino_path in tqdm(sino_paths, desc=f"DPS [{SPLIT}-{n_angles} deg]"):
        rel_path = sino_path.relative_to(input_sino_dir)
        local_path = local_out_dir / rel_path
        drive_path = drive_out_dir / rel_path

        if drive_path.exists():   # gia' su Drive: salta (ripresa dopo disconnessione)
            continue

        local_path.parent.mkdir(parents=True, exist_ok=True)

        try:
            sinogram = np.load(sino_path)
            y_delta = torch.from_numpy(sinogram).float().unsqueeze(0).unsqueeze(0)   # CPU

            x_sol, _ = dps_reconstruct(
                y_delta, K,
                n_iters=N_ITERS, data_weight=data_weight, step_size=step_size, noise_scale=noise_scale,
            )

            np.save(local_path, x_sol.squeeze().astype(np.float32))

        except Exception as e:
            failures.append((str(sino_path), str(e)))
            continue

        n_since_sync += 1
        if n_since_sync >= SYNC_EVERY:
            sync_to_drive(local_out_dir, drive_out_dir)
            n_since_sync = 0

    sync_to_drive(local_out_dir, drive_out_dir)   # sync finale per questa config

print(f"\nFallimenti totali: {len(failures)}")
for path, err in failures[:5]:
    print(path)
    print(" ->", err)


--> Angoli: 90 | data_weight: 0.01 | step_size: 0.01 | noise_scale: 0.0


DPS [test-90 deg]:   0%|          | 0/327 [00:00<?, ?it/s]

Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/gan_dps_reconstructions/test/90).
Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/gan_dps_reconstructions/test/90).
Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/gan_dps_reconstructions/test/90).
Sincronizzati 27 nuovi file su Drive (/content/drive/MyDrive/gan_dps_reconstructions/test/90).

--> Angoli: 45 | data_weight: 0.01 | step_size: 0.01 | noise_scale: 0.0


DPS [test-45 deg]:   0%|          | 0/327 [00:00<?, ?it/s]

Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/gan_dps_reconstructions/test/45).
Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/gan_dps_reconstructions/test/45).
Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/gan_dps_reconstructions/test/45).
Sincronizzati 27 nuovi file su Drive (/content/drive/MyDrive/gan_dps_reconstructions/test/45).

--> Angoli: 30 | data_weight: 0.1 | step_size: 0.001 | noise_scale: 0.02


DPS [test-30 deg]:   0%|          | 0/327 [00:00<?, ?it/s]

Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/gan_dps_reconstructions/test/30).
Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/gan_dps_reconstructions/test/30).
Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/gan_dps_reconstructions/test/30).
Sincronizzati 27 nuovi file su Drive (/content/drive/MyDrive/gan_dps_reconstructions/test/30).

--> Angoli: 15 | data_weight: 0.1 | step_size: 0.001 | noise_scale: 0.02


DPS [test-15 deg]:   0%|          | 0/327 [00:00<?, ?it/s]

Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/gan_dps_reconstructions/test/15).
Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/gan_dps_reconstructions/test/15).
Sincronizzati 100 nuovi file su Drive (/content/drive/MyDrive/gan_dps_reconstructions/test/15).
Sincronizzati 27 nuovi file su Drive (/content/drive/MyDrive/gan_dps_reconstructions/test/15).

Fallimenti totali: 0


## Cosa e' stato generato
Su Drive, in `gan_dps_reconstructions/test/<n_angoli>/<paziente>/<slice>.npy`, trovi le ricostruzioni GAN+DPS per le 4 configurazioni angolari, sul solo test set (stesso schema di TV/Weighted TV/UNet).

**Non ancora fatto** (rimandato apposta, stesso schema degli altri metodi): calcolo di PSNR/SSIM contro la ground truth, confronto visivo, e -- se i risultati non sono soddisfacenti -- tuning di `data_weight`, `step_size`, `noise_scale`, `n_iters` su un campione di validation.